In [1]:
"""Colab (T4) companion: do pretrained transformers behave like the local detectors?

Paste this whole file into one Colab cell and run it. It will ask you to upload two
files from the project, fine-tune one or more transformers, run the identical tests the
local models went through, and hand you a results CSV to drop back into logs/.

Upload these when prompted:
    data/manifest.csv          (3 MB)
    logs/cross_context.csv     (2 MB, optional but it enables the context test)

Three questions are asked of every model, matching ml/evaluate.py and ml/execution_auc.py
exactly so the numbers are comparable with the local table:

  1. AUC(attack vs benign)    the task the literature reports
  2. AUC(executes vs inert)   among attack strings only, does the score track real
                              exploitability? Local models all sat at 0.38 to 0.47.
  3. shortcut probe           rename the marker identifiers, keep structure and
                              obfuscation, and see how much recall survives

Runtime on a T4: roughly 3 to 6 minutes per model.
"""

# ----------------------------------------------------------------------------- setup
SETUP = r'''
!pip -q install "transformers>=4.40" "torch" "scikit-learn" "pandas" 2>/dev/null
'''

import os
import sys

MODELS = [
    "microsoft/codebert-base",      # pretrained on real code: the headline comparison
    "distilroberta-base",           # smaller general-text model, cheap control
    # "microsoft/graphcodebert-base",
    # "google/canine-s",            # character-level: tests the tokenisation question
]
EPOCHS = 4
BATCH = 16
MAX_LEN = 256
LR = 2e-5
SEED = 42

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression  # noqa: F401  (kept for parity)
from sklearn.metrics import roc_auc_score

torch.manual_seed(SEED)
np.random.seed(SEED)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEV}  ({torch.cuda.get_device_name(0) if DEV.type == 'cuda' else 'no GPU'})")


# ------------------------------------------------------------------------ load inputs
def get_files():
    """Colab upload, or read from the working directory if already present."""
    need = {"manifest.csv": None, "cross_context.csv": None}
    for n in list(need):
        for cand in (n, f"data/{n}", f"logs/{n}"):
            if os.path.exists(cand):
                need[n] = cand
    if need["manifest.csv"] is None:
        try:
            from google.colab import files
            print("Upload data/manifest.csv (and optionally logs/cross_context.csv)")
            up = files.upload()
            for k in up:
                if "manifest" in k:
                    need["manifest.csv"] = k
                if "cross_context" in k:
                    need["cross_context.csv"] = k
        except ImportError:
            sys.exit("manifest.csv not found and not running in Colab")
    return need


paths = get_files()
m = pd.read_csv(paths["manifest.csv"], keep_default_na=False)
m["label_attack_like"] = m.label_attack_like.astype(int)
print(f"manifest: {len(m)} rows")

cross = None
if paths["cross_context.csv"]:
    cross = pd.read_csv(paths["cross_context.csv"], keep_default_na=False)
    cross = cross[cross.note == ""]
    cross["ex"] = ((cross.fired_load == 1) | (cross.fired_interact == 1)).astype(int)
    print(f"cross_context: {len(cross)} rows")

# Train on ORIGINALS only, exactly like the local baseline. Variants never enter training.
train = m[(m.split == "train") & (m.label_kind == "original")]
val = m[(m.split == "val") & (m.label_kind == "original")]
assert not (set(train.group_id) & set(val.group_id)), "group leak train/val"
print(f"train {len(train)}  val {len(val)}")


# ------------------------------------------------------------------- shortcut renaming
def _hex(s):
    return "".join(f"\\x{ord(c):02x}" for c in s)


RENAMES = [
    ("'__xss'+'_fired'", "'__probe'+'_flag'"),
    (_hex("__xss_fired"), _hex("__probe_flag")),
    (".__xss_f&#105;r&#101;d=", ".__probe_fl&#97;g="),
    ("&#119;&#105;nd&#111;&#119;", "s&#101;lf"),
    ("__xss_fired", "__probe_flag"),
    ("window", "self"),
]


def rename_marker(t):
    for a, b in RENAMES:
        t = t.replace(a, b)
    return t


# --------------------------------------------------------------------------- training
def finetune(model_name):
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_name)
    net = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(DEV)

    X, y = train.norm_text.tolist(), train.label_attack_like.values
    opt = torch.optim.AdamW(net.parameters(), lr=LR)
    total = max(1, (len(X) // BATCH) * EPOCHS)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=total, pct_start=0.1)
    rng = np.random.default_rng(SEED)
    net.train()
    step = 0
    for ep in range(EPOCHS):
        order = rng.permutation(len(X))
        run = torch.zeros((), device=DEV)      # keep the loss on-device; syncing every
        for i in range(0, len(X), BATCH):      # batch is what crippled this on MPS
            idx = order[i:i + BATCH]
            enc = tok([X[j] for j in idx], truncation=True, max_length=MAX_LEN,
                      padding=True, return_tensors="pt").to(DEV)
            out = net(**enc, labels=torch.tensor(y[idx], device=DEV))
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
            if step < total - 1:
                sched.step()
            step += 1
            opt.zero_grad()
            run += out.loss.detach() * len(idx)
        print(f"    epoch {ep + 1}/{EPOCHS}  loss {run.item() / len(X):.4f}")
    return tok, net


@torch.no_grad()
def score(tok, net, texts, bs=64):
    net.eval()
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], truncation=True, max_length=MAX_LEN,
                  padding=True, return_tensors="pt").to(DEV)
        out.append(F.softmax(net(**enc).logits, -1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)


def pick_threshold(probs, y):
    """Max F1 on validation, ties broken toward fewer false positives. Same rule as local."""
    uniq = np.unique(probs)
    cands = np.unique((uniq[:-1] + uniq[1:]) / 2) if len(uniq) > 1 else np.array([0.5])
    best = (-1, 0.5)
    for t in cands:
        p = (probs >= t).astype(int)
        tp = ((p == 1) & (y == 1)).sum(); fp = ((p == 1) & (y == 0)).sum()
        fn = ((p == 0) & (y == 1)).sum()
        pr = tp / (tp + fp) if tp + fp else 0
        rc = tp / (tp + fn) if tp + fn else 0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0
        if (f1, -fp) > (best[0], -1e9):
            best = (f1, float(t))
    return best[1]


# ------------------------------------------------------------------------- evaluation
test = m[m.split == "test"]
att = test[test.label_kind == "attack_variant"].copy()
att["ex"] = ((att.browser_confirmed.astype(str) == "1") |
             (att.browser_confirmed_with_interaction.astype(str) == "1")).astype(int)
att_d = att.drop_duplicates("norm_text")                 # distinct strings, not rows
ben_d = test[test.label_kind == "fpr_probe"].drop_duplicates("norm_text")
orig_t = test[test.label_kind == "original"]

rows = []
for name in MODELS:
    print(f"\n=== {name}")
    tok, net = finetune(name)

    thr = pick_threshold(score(tok, net, val.norm_text.tolist()), val.label_attack_like.values)
    s_att = score(tok, net, att_d.norm_text.tolist())
    s_ben = score(tok, net, ben_d.norm_text.tolist())
    s_ren = score(tok, net, [rename_marker(t) for t in att_d.norm_text])
    s_o = score(tok, net, orig_t.norm_text.tolist())

    auc_task = roc_auc_score(np.r_[np.ones(len(s_att)), np.zeros(len(s_ben))], np.r_[s_att, s_ben])
    auc_exec = roc_auc_score(att_d.ex.values, s_att)
    rec_orig = float((s_o[orig_t.label_attack_like.values == 1] >= thr).mean())
    rec_var = float((s_att >= thr).mean())
    rec_ren = float((s_ren >= thr).mean())

    r = dict(model=name, threshold=round(thr, 4),
             auc_attack_vs_benign=round(auc_task, 3),
             auc_execute_vs_inert=round(auc_exec, 3),
             recall_ordinary=round(rec_orig, 3),
             recall_variants=round(rec_var, 3),
             recall_after_rename=round(rec_ren, 3),
             shortcut_drop=round(rec_var - rec_ren, 3),
             n_exec=int(att_d.ex.sum()), n_inert=int((1 - att_d.ex).sum()))

    # optional: does the model separate execution when the SINK is crossed in?
    if cross is not None:
        cte = cross[cross.split == "test"]
        s_c = score(tok, net, cte.norm_text.tolist())
        r["auc_execute_crossed_payload_only"] = round(roc_auc_score(cte.ex.values, s_c), 3)
        s_cs = score(tok, net, [f"[{s}] {t}" for s, t in zip(cte.sink, cte.norm_text)])
        r["auc_execute_crossed_with_sink"] = round(roc_auc_score(cte.ex.values, s_cs), 3)

    rows.append(r)
    for k, v in r.items():
        print(f"    {k:<36} {v}")

    del net
    torch.cuda.empty_cache()

res = pd.DataFrame(rows)
res.to_csv("table5_transformers.csv", index=False)
print("\n" + "=" * 90)
print(res.to_string(index=False))
print("=" * 90)
print("\nLocal models for comparison (AUC execute vs inert):")
print("  char_lr 0.439   char_svm 0.467   word_lr 0.473")
print("  hand_rf 0.388   hand_gb 0.382    hybrid_lr 0.453")
print("\nIf the transformers land in that same band, the limit is the task and not the")
print("model. If they are clearly higher, shallow features were the bottleneck.")

try:
    from google.colab import files
    files.download("table5_transformers.csv")
except Exception:
    print("\nsaved table5_transformers.csv")

device: cuda  (Tesla T4)
Upload data/manifest.csv (and optionally logs/cross_context.csv)


Saving cross_context.csv to cross_context.csv
Saving manifest.csv to manifest.csv
manifest: 6377 rows
cross_context: 5644 rows
train 600  val 200

=== microsoft/codebert-base


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch 1/4  loss 0.4758
    epoch 2/4  loss 0.0573
    epoch 3/4  loss 0.0091
    epoch 4/4  loss 0.0034
    model                                distilroberta-base
    threshold                            0.7058
    auc_attack_vs_benign                 0.995
    auc_execute_vs_inert                 0.536
    recall_ordinary                      1.0
    recall_variants                      1.0
    recall_after_rename                  1.0
    shortcut_drop                        0.0
    n_exec                               325
    n_inert                              307
    auc_execute_crossed_payload_only     0.56
    auc_execute_crossed_with_sink        0.568

                  model  threshold  auc_attack_vs_benign  auc_execute_vs_inert  recall_ordinary  recall_variants  recall_after_rename  shortcut_drop  n_exec  n_inert  auc_execute_crossed_payload_only  auc_execute_crossed_with_sink
microsoft/codebert-base     0.5277                 0.997                 0.581             0.99

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>